# 프로젝트 개요




## Kaggle 제출 가이드 (요약)

1. Kaggle 대회 페이지에서 **Data** 탭의 `train.csv`, `test.csv`, `sample_submission.csv` 구조를 확인합니다.
2. 이 노트북을 실행해 `submission.csv`를 생성합니다.
3. Kaggle 대회 페이지 → **Submit Predictions** → `submission.csv` 업로드
4. 제출 파일 형식은 `sample_submission.csv`와 동일해야 합니다.
   - 컬럼: `id`, `health_condition`
   - `id` 순서와 개수는 `test.csv`와 같아야 합니다.
5. Public Leaderboard 점수는 제출 후 확인합니다.

**왜 Accuracy 대신 Balanced Accuracy를 보나?**
- 현재 데이터는 `at-risk` 클래스 비율이 매우 높아 일반 Accuracy만 보면 다수 클래스에 치우친 모델도 높게 나올 수 있습니다.
- `balanced_accuracy_score`는 **각 클래스 recall의 평균**이라 불균형 다중분류에서 더 해석하기 좋습니다.
- scikit-learn 문서도 불균형 분류에서 balanced accuracy 사용을 안내합니다.


## 프로젝트 가이드라인
- 7월 15일 수요일
  + 3교시까지 개인별 or 팀별 / 개인 점수 제출
  + 4교시까지 발표자료 제출 (PPT)
- 발표자료에 반드시 들어갈 내용
  + 발표자료 간지포함(30페이지 내)
    - 자세한 부분은 모두 부록으로 치환
  + 캐글 대회 소개 및 데이터 정의서 정리
  + 프로젝트 정의 (SCQA), 왜 머신러닝 프로젝트가 필요하며, 과정 도식화
  + 프로젝트 수행
    - 탐색적 자료분석 (시각화 및 통계 검정)
    - 머신러닝 모델 비교표 (최소 10개 모델 수행)
      + 시간 측정 필수 (시간 대비 평가지표 비교하는 시각화 필수)
    - 머신러닝 아키텍쳐 (Pipeline 포함)
    - 최종 제출 점수 (ID와 점수 모두 표시되도록 캡쳐)
  + 액션플랜
    - 개발된 모델을 어떤 형태로 프로그램화 할 것인지 개념화
  + 한계점 및 향후 계획
- 프로젝트 조언
  + 절대 바이브코딩 (코드 입력하면서 하기엔 시간이 부족합니다!)
  + 전체 데이터 활용하여 모델 비교표 만들면 시간내에 못합니다. 샘플링 적용 필수
  + 프로젝트의 전체 흐름을 이해하는 것이 이번 미니 프로젝트의 목적
  + 7월 중순 이후, 의미있는 개인 프로젝트로 잘 정리할 것


# 라이브러리 불러오기

In [1]:
# 필요한 라이브러리 import
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, classification_report

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


# 데이터 불러오기

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ============================================================
# 1. 데이터 로드
# ============================================================
DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/2026/이어드림스쿨6기/dataset/kaggle_student_classification/'

train_df = pd.read_csv(DATA_PATH + 'train.csv')
test_df = pd.read_csv(DATA_PATH + 'test.csv')
sample_submission = pd.read_csv(DATA_PATH + 'sample_submission.csv')

print('train shape:', train_df.shape)
print('test shape :', test_df.shape)
print('sample_submission shape:', sample_submission.shape)
print()
print('target classes:')
print(train_df['health_condition'].value_counts())
print()
print('target ratio:')
print(train_df['health_condition'].value_counts(normalize=True).round(4))
print()
train_df.head()


train shape: (690088, 15)
test shape : (295753, 14)
sample_submission shape: (295753, 2)

target classes:
health_condition
at-risk      592561
unhealthy     57724
fit           39803
Name: count, dtype: int64

target ratio:
health_condition
at-risk      0.8587
unhealthy    0.0836
fit          0.0577
Name: proportion, dtype: float64



,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male


# 변수 설명

Kaggle 데이터셋 설명을 바탕으로 각 변수를 아주 간단히 정리하면 아래와 같습니다.

| 변수명 | 한글 설명 |
|---|---|
| `sleep_duration` | 하루 평균 수면 시간 |
| `heart_rate` | 안정 시 심박수 또는 평균 심박수 |
| `bmi` | 체질량지수 |
| `calorie_expenditure` | 하루 칼로리 소모량 |
| `step_count` | 하루 걸음 수 |
| `exercise_duration` | 운동 시간 |
| `water_intake` | 하루 물 섭취량 |
| `diet_type` | 식단 유형 (`veg`, `non-veg`, `balanced` 등) |
| `stress_level` | 스트레스 수준 (`low`, `medium`, `high`) |
| `sleep_quality` | 수면의 질 (`poor`, `average`, `good`) |
| `physical_activity_level` | 신체 활동 수준 (`sedentary`, `moderate`, `active`) |
| `smoking_alcohol` | 흡연/음주 여부 또는 빈도 |
| `gender` | 성별 |
| `health_condition` | 예측 대상 건강 상태 (`fit`, `at-risk`, `unhealthy`) |

- 일반적인 건강 위험도 분류 방식을 기준으로 정리해 드릴게요.
- 타겟 클래스 (건강 상태 등급) 의미

| 클래스 | 의미 | 일반적으로 대응되는 특성 |
|---|---|---|
| **Fit** | 균형 잡히고 건강한 상태 | 규칙적인 운동, 충분한 수면(7~9시간), 균형 잡힌 식습관, 낮은 스트레스 수준, 정상 체중(BMI) |
| **At-Risk** | 중간 수준의 건강 우려가 있는 상태 | 불규칙한 생활 패턴(수면 부족·과다, 운동 부족), 스트레스 증가, 식습관 일부 불균형 — 아직 심각하지는 않지만 개선이 필요한 경계 상태 |
| **Unhealthy** | 높은 수준의 건강 위험이 있는 상태 | 만성적 수면 부족, 신체 활동 거의 없음, 흡연/음주/패스트푸드 등 위험 행동 빈번, 높은 스트레스·불안 지표, 비정상 체중 등 복합적 위험 요인 |


In [4]:
# ============================================================
# 2. feature / target 분리
# ============================================================
target_col = 'health_condition'
id_col = 'id'

feature_cols = [
    'sleep_duration',
    'heart_rate',
    'bmi',
    'calorie_expenditure',
    'step_count',
    'exercise_duration',
    'water_intake',
    'diet_type',
    'stress_level',
    'sleep_quality',
    'physical_activity_level',
    'smoking_alcohol',
    'gender',
]

X = train_df.loc[:, feature_cols].copy()
y = train_df.loc[:, target_col].copy()
X_test_kaggle = test_df.loc[:, feature_cols].copy()
test_ids = test_df.loc[:, id_col].copy()

print('X shape:', X.shape)
print('y shape:', y.shape)
print('X_test_kaggle shape:', X_test_kaggle.shape)
print()
print('결측치 개수 (train):')
print(X.isnull().sum())


X shape: (690088, 13)
y shape: (690088,)
X_test_kaggle shape: (295753, 13)

결측치 개수 (train):
sleep_duration             75999
heart_rate                  7833
bmi                        13898
calorie_expenditure        52853
step_count                 13916
exercise_duration           6901
water_intake               43477
diet_type                   6901
stress_level               82811
sleep_quality              58331
physical_activity_level    36621
smoking_alcohol            28582
gender                     21373
dtype: int64


# train / validation 분리 (층화 추출)

In [5]:
# ============================================================
# 3. train / validation 분리 (층화 추출)
# ============================================================
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f'학습 데이터: {len(X_train):,}행')
print(f'검증 데이터: {len(X_valid):,}행')
print()
print('[전체 y 비율]')
print(y.value_counts(normalize=True).sort_index().round(4))
print()
print('[train y 비율]')
print(y_train.value_counts(normalize=True).sort_index().round(4))
print()
print('[validation y 비율]')
print(y_valid.value_counts(normalize=True).sort_index().round(4))


학습 데이터: 552,070행
검증 데이터: 138,018행

[전체 y 비율]
health_condition
at-risk      0.8587
fit          0.0577
unhealthy    0.0836
Name: proportion, dtype: float64

[train y 비율]
health_condition
at-risk      0.8587
fit          0.0577
unhealthy    0.0836
Name: proportion, dtype: float64

[validation y 비율]
health_condition
at-risk      0.8587
fit          0.0577
unhealthy    0.0836
Name: proportion, dtype: float64


# 파이프라인 정의

## 수치형 범주형 feature 정의

In [6]:
# ============================================================
# 4. 수치형 / 범주형 feature 정의
# ============================================================
numeric_features = [
    'sleep_duration',
    'heart_rate',
    'bmi',
    'calorie_expenditure',
    'step_count',
    'exercise_duration',
    'water_intake',
]

categorical_features = [
    'diet_type',
    'stress_level',
    'sleep_quality',
    'physical_activity_level',
    'smoking_alcohol',
    'gender',
]

print('수치형 feature:', numeric_features)
print('범주형 feature:', categorical_features)


수치형 feature: ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']
범주형 feature: ['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']


## ColumnTransformer + Pipeline 구성

In [7]:
# ============================================================
# 5. ColumnTransformer + Pipeline 구성
# ============================================================

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
])

logistic_model = LogisticRegression(
    max_iter=1000,
    solver='saga',
    random_state=RANDOM_STATE,
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', logistic_model),
])

print('Pipeline ready')


Pipeline ready


# 과적합 확인용 교차 검증

## 과적합은 어떻게 확인할까?

- 과적합 방지용

1. **CV train balanced accuracy**
2. **CV valid balanced accuracy**
3. **hold-out validation balanced accuracy**

해석 기준은 아주 단순하게 보면 됩니다.

- `train 점수 >> valid 점수` 이면 과적합 가능성
- `train 점수 ≈ valid 점수` 이면 과적합이 크지 않을 가능성
- 둘 다 낮으면 과소적합 가능성

즉, 로지스틱 회귀에서도 **train/valid 점수 차이**를 보면 과적합 여부를 점검할 수 있습니다.


## 샘플링의 마법

### 1. 연산량 문제
- `cross_validate`는 `pipeline`을 **5번(n_splits=5) 처음부터 재학습**함
- `GridSearchCV`/`RandomizedSearchCV`까지 곁들이면 파라미터 조합 수만큼 추가로 곱해짐
- 비교:
  - 5,000행 × 5 fold → fold당 학습 데이터 약 4,000행
  - 690,088행 × 5 fold → fold당 학습 데이터 약 552,000행 (**약 138배 차이**)
- 트리 기반 모델(RandomForest, GBM 등)이나 무거운 전처리(OneHotEncoder + 고차원 피처, TF-IDF 등)가 있으면 전체 데이터 5-fold CV는 수십 분~수 시간 소요 가능
- 노트북에서 모델/피처를 반복 실험해야 하는 단계에서는 이 비용이 치명적

### 2. "빠른 확인용"이라는 목적
- 이 단계의 목적은 파이프라인이 대략 잘 작동하는지, train-valid gap(과적합)이 심한지 **빠르게 스크리닝**하는 것
- 최종 성능 정밀 측정이 목적이 아님
- 최종 제출 모델은 이후 **전체 train 데이터로 다시 학습**함 (본게임 전 리허설 역할)

### 3. 층화추출(stratify)로 대표성 확보
- `y_train.groupby(y_train, ...).sample(...)`로 각 클래스 비율을 유지한 채 샘플링
- 무작위 5,000개보다 클래스 불균형 문제를 CV 단계에서도 어느 정도 반영 가능

---

### 이 접근의 한계 (트레이드오프)

| 항목 | 영향 |
|---|---|
| **분산 증가** | 5,000행 기준이라 fold별 점수 변동폭(`valid_std`)이 실제보다 크게 나올 수 있음 |
| **소수 클래스 표현 부족** | 클래스가 매우 불균형하면(예: 0.1% 미만) 5,000행 샘플엔 fold마다 몇 개 안 들어가 점수 불안정 |
| **점수 신뢰도** | 5,000행 CV 점수는 참고용, "진짜 일반화 성능"과 차이 있을 수 있음 (데이터가 클수록 학습 곡선이 우상향하는 경향이 있어, 전체 데이터 학습 시 성능이 더 좋게 나올 가능성 높음) |
| **모델 간 상대 비교엔 유용** | 절대 성능보다 "모델 A vs 모델 B", "피처 셋 변경 전후" 같은 **상대적 비교**에는 여전히 유용 |

---

### 요약
- **속도(연산 비용)** 때문에 샘플링 → 실험 반복(모델 선택, 하이퍼파라미터 튜닝 방향 결정)을 빠르게 하기 위함
- **최종 성능 검증**은 이 CV 점수가 아니라, 전체 데이터로 학습한 뒤 validation set(`X_valid`, `y_valid`)으로 평가하는 결과를 신뢰해야 함
- 690k 전체로 CV를 돌려도 시간이 감당되고 더 신뢰도 높은 CV 점수가 필요하다면:
  - `CV_SAMPLE_SIZE`를 늘리기 (예: 50,000~100,000)
  - `n_splits`을 줄이기 (예: 3-fold)
  - 전체 데이터로 CV를 돌리기

In [8]:
# ============================================================
# 6. StratifiedKFold 교차 검증
# ============================================================
# 데이터가 매우 크므로, 교차 검증은 층화 샘플 5,000행으로 빠르게 확인합니다.
# 최종 제출용 모델은 뒤에서 전체 train 데이터로 다시 학습합니다.

import time

CV_SAMPLE_SIZE = 5_000
cv_sample_n = min(CV_SAMPLE_SIZE, len(y_train))

cv_sample_idx = (
    y_train.groupby(y_train, group_keys=False)
    .apply(lambda s: s.sample(frac=cv_sample_n / len(y_train), random_state=RANDOM_STATE))
    .index
)

X_cv = X_train.loc[cv_sample_idx]
y_cv = y_train.loc[cv_sample_idx]

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

scoring = {
    'balanced_accuracy': 'balanced_accuracy',
    'f1_macro': 'f1_macro',
}

# ---- 시간 측정 시작 ----
start_time = time.time()

cv_results = cross_validate(
    pipeline,
    X_cv,
    y_cv,
    cv=cv,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1,
)

elapsed_time = time.time() - start_time
# ---- 시간 측정 종료 ----

cv_summary = pd.DataFrame({
    'metric': ['balanced_accuracy', 'f1_macro'],
    'train_mean': [
        cv_results['train_balanced_accuracy'].mean(),
        cv_results['train_f1_macro'].mean(),
    ],
    'valid_mean': [
        cv_results['test_balanced_accuracy'].mean(),
        cv_results['test_f1_macro'].mean(),
    ],
    'valid_std': [
        cv_results['test_balanced_accuracy'].std(),
        cv_results['test_f1_macro'].std(),
    ],
    'gap_train_minus_valid': [
        cv_results['train_balanced_accuracy'].mean() - cv_results['test_balanced_accuracy'].mean(),
        cv_results['train_f1_macro'].mean() - cv_results['test_f1_macro'].mean(),
    ],
})

print(f'CV sample size: {len(X_cv):,}')
print(f'CV 소요 시간: {elapsed_time:.2f}초 ({elapsed_time/60:.2f}분)')
cv_summary.round(4)

CV sample size: 4,999
CV 소요 시간: 17.93초 (0.30분)


,metric,train_mean,valid_mean,valid_std,gap_train_minus_valid
0,balanced_accuracy,0.8076,0.7975,0.0126,0.0102
1,f1_macro,0.8544,0.8432,0.0112,0.0111


### 결과 해석

- 좋은 신호
  + **train-valid gap이 매우 작음** (약 0.01 수준) → 과적합 징후가 거의 없음. 5,000행이라는 작은 샘플로도 파이프라인이 안정적으로 일반화되고 있다는 뜻
  + **valid_std도 낮음** (0.011~0.013) → fold 간 점수 편차가 크지 않아, 이 5,000행 샘플 CV 결과가 어느 정도 신뢰할 만함
  + balanced_accuracy(~ 0.80)와 f1_macro(~ 0.84) 모두 준수한 수준으로, 클래스 불균형을 고려해도 무난한 성능

# validation 세트 평가

In [9]:
# ============================================================
# 7. validation 세트 평가
# ============================================================
import time

start_time = time.time()
pipeline.fit(X_train, y_train)
fit_time = time.time() - start_time

start_time = time.time()
y_train_pred = pipeline.predict(X_train)
train_pred_time = time.time() - start_time

start_time = time.time()
y_valid_pred = pipeline.predict(X_valid)
valid_pred_time = time.time() - start_time

train_bal_acc = balanced_accuracy_score(y_train, y_train_pred)
valid_bal_acc = balanced_accuracy_score(y_valid, y_valid_pred)

print(f'Fit 소요 시간           : {fit_time:.2f}초 ({fit_time/60:.2f}분)')
print(f'Train 예측 소요 시간    : {train_pred_time:.2f}초')
print(f'Valid 예측 소요 시간    : {valid_pred_time:.2f}초')
print()
print('Train Balanced Accuracy     :', round(train_bal_acc, 4))
print('Validation Balanced Accuracy:', round(valid_bal_acc, 4))
print('Gap (train - valid)         :', round(train_bal_acc - valid_bal_acc, 4))
print()
print('--- Validation Classification Report ---')
print(classification_report(y_valid, y_valid_pred))

Fit 소요 시간           : 204.55초 (3.41분)
Train 예측 소요 시간    : 1.20초
Valid 예측 소요 시간    : 0.35초

Train Balanced Accuracy     : 0.8129
Validation Balanced Accuracy: 0.8137
Gap (train - valid)         : -0.0008

--- Validation Classification Report ---
              precision    recall  f1-score   support

     at-risk       0.96      0.98      0.97    118512
         fit       0.88      0.73      0.80      7961
   unhealthy       0.89      0.72      0.80     11545

    accuracy                           0.95    138018
   macro avg       0.91      0.81      0.86    138018
weighted avg       0.95      0.95      0.95    138018



# 과적합 확인

In [10]:
# ============================================================
# 8. 과적합 여부 해석
# ============================================================
cv_train_bal = cv_results['train_balanced_accuracy'].mean()
cv_valid_bal = cv_results['test_balanced_accuracy'].mean()

overfit_check = pd.DataFrame({
    'check_point': [
        'CV train balanced_accuracy',
        'CV valid balanced_accuracy',
        'Hold-out train balanced_accuracy',
        'Hold-out valid balanced_accuracy',
    ],
    'score': [
        cv_train_bal,
        cv_valid_bal,
        train_bal_acc,
        valid_bal_acc,
    ],
})

print(overfit_check.round(4))
print()

if (train_bal_acc - valid_bal_acc) < 0.02 and (cv_train_bal - cv_valid_bal) < 0.02:
    print('해석: train/valid 차이가 크지 않아, 현재 로지스틱 회귀 기준으로 과적합이 크다고 보기는 어렵습니다.')
elif (train_bal_acc - valid_bal_acc) < 0.05 and (cv_train_bal - cv_valid_bal) < 0.05:
    print('해석: 약한 수준의 과적합 가능성은 있지만, 심한 차이는 아닙니다.')
else:
    print('해석: train 점수가 valid 점수보다 꽤 높아 과적합 가능성을 점검할 필요가 있습니다.')


                        check_point   score
0        CV train balanced_accuracy  0.8076
1        CV valid balanced_accuracy  0.7975
2  Hold-out train balanced_accuracy  0.8129
3  Hold-out valid balanced_accuracy  0.8137

해석: train/valid 차이가 크지 않아, 현재 로지스틱 회귀 기준으로 과적합이 크다고 보기는 어렵습니다.


# 최종 학습

In [11]:
# ============================================================
# 9. 전체 train 데이터로 최종 학습
# ============================================================
import time

X_full = train_df.loc[:, feature_cols].copy()
y_full = train_df.loc[:, target_col].copy()

final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', logistic_model),
])

start_time = time.time()
final_pipeline.fit(X_full, y_full)
elapsed_time = time.time() - start_time

print('최종 모델 학습 완료')
print('학습 데이터 수:', len(X_full))
print(f'학습 소요 시간: {elapsed_time:.2f}초 ({elapsed_time/60:.2f}분)')

최종 모델 학습 완료
학습 데이터 수: 690088
학습 소요 시간: 237.51초 (3.96분)


# 예측

In [12]:
# ============================================================
# 10. test 예측 및 submission.csv 생성
# ============================================================
test_pred = final_pipeline.predict(X_test_kaggle)

submission = pd.DataFrame({
    id_col: test_ids,
    target_col: test_pred,
})

submission_path = 'submission_1th.csv'
submission.to_csv(submission_path, index=False)

print('submission shape:', submission.shape)
print('저장 경로:', submission_path)
print()
print('예측 클래스 분포:')
print(submission[target_col].value_counts())
print()
print('sample_submission 과 컬럼 비교:')
print('submission columns     :', submission.columns.tolist())
print('sample_submission cols :', sample_submission.columns.tolist())
print()
submission.head()


submission shape: (295753, 2)
저장 경로: submission_1th.csv

예측 클래스 분포:
health_condition
at-risk      261769
unhealthy     19831
fit           14153
Name: count, dtype: int64

sample_submission 과 컬럼 비교:
submission columns     : ['id', 'health_condition']
sample_submission cols : ['id', 'health_condition']



,id,health_condition
0,690088,unhealthy
1,690089,at-risk
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy


# 참고 링크

- balanced accuracy 문서: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.balanced_accuracy_score.html
- 데이터셋 설명 페이지: https://www.kaggle.com/datasets/ziya07/college-student-health-behavior-dataset
